In [ ]:
import torch
from diffusion.approaches.matching.prob_paths import (
    GaussianCondProbPath,
    LinearAlpha,
    LinearBeta,
)
from diffusion.approaches.matching.flow_trainer import FlowTrainer
from diffusion.sampleables.mnist_sampleable import MNISTSampleable
from diffusion.architectures.backbones.res_unet import ResUnet

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(device)

mps


In [ ]:
sampeable = MNISTSampleable(train=True)
val_sampeable = MNISTSampleable(train=False)

path = GaussianCondProbPath(
    p_data=sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

val_path = GaussianCondProbPath(
    p_data=val_sampeable,
    p_simple_shape=sampeable.shape,
    alpha=LinearAlpha(),
    beta=LinearBeta(),
).to(device)

backbone = ResUnet(
    in_channels=1,
    channels=[16, 32, 64],
    num_classes=sampeable.num_classes,
    t_dim=64,
    y_dim=32,
    cond_dim=64,
).to(device)

trainer = FlowTrainer(
    path=path,
    val_path=val_path,
    backbone=backbone,
    null_class=sampeable.num_classes,
)

In [ ]:
state_dict = trainer.train(
    num_epochs=15,
    device=device,
    lr=3e-4,
    batch_size=64,
    steps_per_epoch=500,
    validate=True,
)

In [ ]:
torch.save(backbone.state_dict(), "./models/backbone_flow_bili.pt")